# Milestone 2 — EDA, without weather

Looking at the data before modelling anything. Seven questions, in order, plus one
housekeeping step first.

**What this notebook does not do.** No model is fitted and no forecast is scored.
Section 6 reports how far the series moves from one day and one week to the next,
which is a property of the data rather than a baseline result — there is no rolling
origin and no train/test split behind it. The seasonal-naive threshold is set in
Milestone 4, on top of the framework built in Milestone 3. Spec section 8 puts the
framework before the models deliberately, so that no number produced now can become
something a later decision anchors on.

Weather is absent on purpose. Track A in spec section 4 — the operationally honest
one, and the headline result — uses no weather at all. Weather becomes necessary at
Milestone 7, not here.

## Setup

In [ ]:
!pip install -q polars duckdb holidays

In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted.")
else:
    print("Not using Drive. Data is rebuilt each session.")

In [ ]:
import os, pathlib, subprocess

REPO_URL = "https://github.com/ethantosc/ieso-demand-forecast.git"
REPO_DIR = pathlib.Path("/content/ieso-demand-forecast")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
import pathlib, shutil

if USE_DRIVE:
    DATA_ROOT = pathlib.Path("/content/drive/MyDrive/ieso-demand-forecast-data")
else:
    DATA_ROOT = pathlib.Path("/content/ieso-demand-forecast-data")

for sub in ("raw/demand", "staging", "curated"):
    (DATA_ROOT / sub).mkdir(parents=True, exist_ok=True)
    link = pathlib.Path("data") / sub
    link.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        if [p for p in link.rglob("*") if p.suffix.lower() in {".csv", ".parquet"}]:
            raise SystemExit(f"data/{sub} holds data files. Decide before re-running.")
        shutil.rmtree(link)
    link.symlink_to(DATA_ROOT / sub)

print("store:", DATA_ROOT, "| persistent:", USE_DRIVE)

Rebuild the Parquet if it is not already there. With `USE_DRIVE = False` this runs
every session and takes about a minute.

In [ ]:
import subprocess, pathlib

if pathlib.Path("data/staging/demand_hourly.parquet").exists():
    print("staging Parquet already present")
else:
    subprocess.run(["python", "-m", "src.ingest.fetch_demand"], check=True)
    subprocess.run(["python", "-m", "src.ingest.demand"], check=True)

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from src.analysis import eda

plt.rcParams.update({"figure.figsize": (11, 4.2), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 10})
pl.Config.set_tbl_rows(40)

raw = eda.load()
print(f"{raw.height:,} rows, {raw['local_date'].min()} .. {raw['local_date'].max()}")

## 0. The 2003 blackout window

Milestone 1 flagged seven hours below 5,000 MW on 14 August 2003. Those are the
hours the grid was down. They are not the whole story: Ontario then ran an
emergency conservation period, and demand stayed below normal afterwards at levels
comfortably above 5,000 MW — low enough to distort an average daily profile, high
enough to pass through the Milestone 1 checks unnoticed.

Excluding only the 14th would leave those days in, quietly dragging down every
profile and difference distribution in this notebook.

So the window is measured rather than guessed. Each day is expressed as a fraction
of its own year's mean, which removes the long-run trend, then compared against the
median of the same calendar date in 2002 and 2004–2006. Days more than 10% below
that expectation count as suppressed, and the run stops at the first normal day.

**Limitation.** With no weather data, this cannot distinguish conservation from a
cool spell. The window is approximate and leans towards excluding too much. It goes
into `data_quality.md` on those terms.

In [ ]:
window = eda.blackout_window(raw)
excluded = eda.exclusion_dates(window)

fig, ax = plt.subplots()
ax.bar([str(r["local_date"])[5:] for r in window.head(16).iter_rows(named=True)],
       [r["deficit"] * 100 for r in window.head(16).iter_rows(named=True)],
       color=["#c0392b" if r["suppressed"] else "#95a5a6"
              for r in window.head(16).iter_rows(named=True)])
ax.axhline(10, ls="--", c="k", lw=1, label="10% threshold")
ax.set_ylabel("% below expected"); ax.set_title("August 2003: demand deficit vs neighbouring years")
ax.legend(); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

print(f"excluded {len(excluded)} days: {excluded[0]} .. {excluded[-1]}")
window.head(12)

In [ ]:
# Nothing is deleted. The flag stays on the table and the analyses filter on it.
flagged = eda.mark_excluded(raw, excluded)
df = eda.clean(flagged)
print(f"{df.height:,} rows in play ({raw.height - df.height} set aside)")

## 1. Does the clock survive a test that could fail it?

Milestone 1 concluded the raw files are fixed EST. Two of the checks behind that
were circular — they were computed from the assumption they appeared to confirm.
The non-circular evidence was the raw row counts and the timing of the 2003
blackout.

This is a third check, and it is falsifiable. The morning load ramp is driven by
alarm clocks, so in a correctly converted series it sits at the same **local** hour
year round. If the raw file were wall-clock time and the ingest had wrongly added
an hour, the ramp would jump by exactly one hour at every DST transition.

`shift` is the change in mean ramp hour across a transition. `control_shift` is the
same measurement three weeks earlier, where no transition happens — that is how
much the ramp naturally drifts between two nearby weeks, and it is the yardstick.

**Read it like this:** if `shift` looks like `control_shift`, the convention holds.
If `shift` approaches 1.0 while the control stays near zero, it does not.

In [ ]:
test = eda.dst_ramp_test(df)

fig, ax = plt.subplots()
ax.axhline(0, c="k", lw=0.8)
ax.axhline(1, c="#c0392b", ls="--", lw=1, label="a one-hour error would sit here")
ax.axhline(-1, c="#c0392b", ls="--", lw=1)
ax.plot(test["transition"], test["control_shift"], "o", ms=5, c="#95a5a6",
        label="control (no transition)")
ax.plot(test["transition"], test["shift"], "o", ms=6, c="#2980b9",
        label="across the transition")
ax.set_ylabel("change in ramp hour"); ax.set_ylim(-1.3, 1.3)
ax.set_title("Morning ramp hour across DST transitions"); ax.legend()
plt.tight_layout(); plt.show()

test.group_by("kind").agg(
    pl.len().alias("n"),
    pl.col("shift").abs().mean().round(3).alias("mean_abs_shift"),
    pl.col("control_shift").abs().mean().round(3).alias("mean_abs_control"),
)

## 2. The daily shape

Mean demand by local hour, split by season and by weekday/weekend. This is the
single most important picture for feature design: whatever the model learns about
hour-of-day has to reproduce these curves.

Watch for whether summer and winter have the **same shape at different heights**,
or genuinely different shapes. If the latter, an hour-of-day feature on its own
cannot carry it and hour must interact with season.

In [ ]:
profile = eda.intraday_profile(df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, weekend, label in zip(axes, [False, True], ["Weekday", "Weekend"]):
    for season, colour in [("summer", "#e67e22"), ("winter", "#2980b9"),
                           ("shoulder", "#7f8c8d")]:
        part = profile.filter((pl.col("season") == season) &
                              (pl.col("is_weekend") == weekend)).sort("local_hour")
        ax.plot(part["local_hour"], part["mean_mw"], label=season, c=colour, lw=2)
    ax.set_title(label); ax.set_xlabel("local hour"); ax.set_xticks(range(0, 24, 3))
axes[0].set_ylabel("mean MW"); axes[0].legend()
plt.tight_layout(); plt.show()

## 3. The weekly shape

How far weekends fall below weekdays, by season. The size of this gap decides
whether a weekend flag is enough or whether each day of the week needs its own
treatment.

In [ ]:
weekly = eda.weekly_profile(df)
names = {1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat", 7: "Sun"}

fig, ax = plt.subplots()
for season, colour in [("summer", "#e67e22"), ("winter", "#2980b9"),
                       ("shoulder", "#7f8c8d")]:
    part = weekly.filter(pl.col("season") == season).sort("weekday")
    ax.plot([names[d] for d in part["weekday"]], part["mean_mw"],
            "o-", label=season, c=colour, lw=2)
ax.set_ylabel("mean MW"); ax.set_title("Demand by day of week"); ax.legend()
plt.tight_layout(); plt.show()
weekly

## 4. When does the year peak?

Milestone 1 found that six of twenty-five annual peaks landed in winter, which
makes spec section 2.2's summer-peaking description too simple. This plots it.

If winter peaks are scattered through the record rather than confined to the early
years, the seasonal error decomposition in spec section 5 is load-bearing: a model
tuned on summer afternoons will be judged on winter evenings too.

In [ ]:
peaks = eda.annual_peaks(df)

fig, ax = plt.subplots()
colours = {"summer": "#e67e22", "winter": "#2980b9", "shoulder": "#7f8c8d"}
ax.scatter(peaks["year"], peaks["peak_month"],
           s=peaks["peak_mw"] / 220, c=[colours[s] for s in peaks["peak_season"]],
           alpha=0.85, edgecolors="white")
ax.set_yticks(range(1, 13)); ax.set_ylabel("month of annual peak")
ax.set_title("When each year peaked (marker size = peak MW)")
plt.tight_layout(); plt.show()

peaks.select("year", "peak_ts_local", "peak_mw", "peak_season", "peak_hour", "mean_mw")

## 5. How stale does old data get?

Spec section 5 says to run both a 3-year rolling window and a full-history
expanding window and compare. This measures what that choice actually costs.

For each evaluation year and each window length W, the demand level is estimated as
the mean over the previous W years, and the error is estimate minus actual. A
negative bias means the estimate came in below what actually happened — old data
under-predicting a rising level.

The trade-off is real in both directions: a short window tracks the level but sees
fewer years of weather variety, a long window is stable but carries a stale level.
This curve gives the first half of that trade a number.

In [ ]:
bias = eda.level_bias_by_window(df)

fig, ax = plt.subplots()
for season, colour in [("all", "#2c3e50"), ("summer", "#e67e22"), ("winter", "#2980b9")]:
    part = bias.filter(pl.col("season") == season).sort("window_years")
    labels = [str(w) if w < 99 else "all" for w in part["window_years"]]
    ax.plot(labels, part["mean_abs_bias_mw"], "o-", label=season, c=colour, lw=2)
ax.set_xlabel("training window (years)"); ax.set_ylabel("mean |level error| MW")
ax.set_title("Cost of using older data to set the demand level"); ax.legend()
plt.tight_layout(); plt.show()

bias.filter(pl.col("season").is_in(["all", "summer", "winter"]))

## 6. How far does the series move?

Absolute change over 24 hours and over 168 hours (same hour last week), by season.

**This is not a baseline score.** No rolling origin, no train/test split, no model.
It is the spread of the series itself, and it is here only to calibrate
expectations: if the median 168-hour change is small, seasonal naive is a hard
threshold to beat and Milestone 6 has real work to do.

The threshold itself gets set in Milestone 4, measured properly on the framework
from Milestone 3. Nothing in this table is that number.

In [ ]:
eda.diff_distribution(df)

## 7. Holidays

Spec section 3 calls for a statutory holiday flag and flags for the day before and
after. Before building those, it is worth seeing which holidays actually move
demand, because they do not all behave alike.

Each holiday is compared against the same weekday in the same month and year, so a
Monday holiday is measured against other Mondays rather than against the weekend
around it.

In [ ]:
import holidays as holidays_pkg

years = df["year"].unique().to_list()
ontario = holidays_pkg.country_holidays("CA", subdiv="ON", years=years)
holiday_names = dict(ontario.items())

effect = eda.holiday_effect(df, set(holiday_names))
summary = eda.holiday_summary(effect, holiday_names)

fig, ax = plt.subplots(figsize=(11, 5))
top = summary.filter(pl.col("n_years") >= 5).sort("mean_delta_pct")
ax.barh(top["holiday"], top["mean_delta_pct"],
        color=["#c0392b" if v < -3 else "#95a5a6" for v in top["mean_delta_pct"]])
ax.axvline(0, c="k", lw=0.8)
ax.set_xlabel("% vs same weekday, same month and year")
ax.set_title("Holiday effect on daily mean demand")
plt.tight_layout(); plt.show()

summary

## What to take from this

Fill in after looking at the plots, then carry the conclusions into
`reports/findings.md` and the Milestone 3 design:

- Does the DST check hold? (Section 1 — if `shift` approaches 1.0 while the control
  stays near zero, stop and raise it; everything downstream depends on this.)
- Does hour-of-day need to interact with season, or is one shape enough? (Section 2)
- Is a weekend flag sufficient, or does each weekday need its own term? (Section 3)
- Which holidays actually matter, and is a single flag enough for all of them?
  (Section 7)
- What does the level-bias curve imply for the window lengths worth backtesting?
  (Section 5)

## Save the tables

The clone disappears with the session. These feed `findings.md` and the Milestone 3
design, so they need to leave `/content/`.

In [ ]:
import pathlib, shutil

out = pathlib.Path("reports/eda")
out.mkdir(parents=True, exist_ok=True)

tables = {
    "blackout_window": window,
    "dst_ramp_test": test,
    "intraday_profile": profile,
    "weekly_profile": weekly,
    "annual_peaks": peaks,
    "level_bias": bias,
    "diff_distribution": eda.diff_distribution(df),
    "holiday_summary": summary,
}
for name, table in tables.items():
    table.write_csv(out / f"{name}.csv")
print("wrote", len(tables), "tables to", out)

if USE_DRIVE:
    dest = DATA_ROOT / "reports" / "eda"
    dest.mkdir(parents=True, exist_ok=True)
    for name in tables:
        shutil.copy(out / f"{name}.csv", dest / f"{name}.csv")
    print("copied to", dest)
else:
    from google.colab import files
    shutil.make_archive("/content/eda_tables", "zip", out)
    files.download("/content/eda_tables.zip")